In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
test_files = fs.glob("datasets/magistermilitum/Tridis/data/test-*.parquet")
print(test_files)

labels = pq.read_table(test_files, columns=["Corpus", "Language"],
filesystem=fs).to_pandas()
print(labels.value_counts())

In [ ]:
table = pq.read_table(
    test_files,
    filesystem=fs,
    filters=[("Corpus", "=", "MLH"), ("Language", "=", "la")],
    )
mlh = table.to_pandas()

print(len(mlh), "lines")
print(mlh.columns.tolist())
print(mlh["text"].head())

In [ ]:
from pathlib import Path
from io import BytesIO
from PIL import Image

out = Path("data/mlh")
out.mkdir(parents=True, exist_ok=True)

names = []
truth = []
for i, row in mlh.iterrows():
    name = f"mlh_{i:04d}"
    image = Image.open(BytesIO(row["image"]))
    image.save(out / f"{name}.png")
    names.append(name)
    truth.append(f"{name} {row['text']}")

(out / "img_list.txt").write_text("\n".join(names) + "\n")
(out / "ground_truth.txt").write_text("\n".join(truth) + "\n")
print("saved", len(names), "lines to", out)

In [ ]:
from IPython.display import display

for name, text in list(zip(names, mlh["text"]))[:3]:
    print(name, "→", text)
    display(Image.open(out / f"{name}.png"))

In [6]:
endp_labels = pq.read_table(
    test_files,
    columns=["Corpus", "Language", "Image_name"],
    filesystem=fs,
    filters=[("Corpus", "=", "E-NDP"), ("Language", "=", "la")],
).to_pandas()

print(len(endp_labels), "lines")
print(endp_labels["Image_name"].head(10))

3335 lines
0    FRAN_0393_06825_L
1    FRAN_0393_08473_L
2    FRAN_0393_08152_L
3    FRAN_0393_03497_L
4    FRAN_0393_04441_L
5    FRAN_0393_07063_L
6      0010_Main_frame
7      0012_Main_frame
8    FRAN_0393_08984_L
9    FRAN_0393_09011_L
Name: Image_name, dtype: str


In [7]:
catmus_files = fs.glob("datasets/CATMuS/medieval/**/*.parquet")
catmus_endp_files = [f for f in catmus_files if "Archives_nationales__LL" in f]
print(len(catmus_endp_files), "files")

catmus_text = pq.read_table(
    catmus_endp_files,
    columns=["text"],
    filesystem=fs,
).to_pandas()["text"]
print(len(catmus_text), "lines")
print(catmus_text.head())

5 files
1837 lines
0                                     I. de soco ꝓ p̃ẜ
1                                           G. fedelli
2                                           I. uoignon
3    Taxata est Emenda nuꝑ plicata ꝑ dñm Ioħem de ...
4                                             T. hocie
Name: text, dtype: str


In [8]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

ns = {"p": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15"}
xml_dir = Path("data/endp/HTR_Ground-Truth/page_xml")

rows = []
for xml_file in sorted(xml_dir.glob("*.xml")):
    root = ET.parse(xml_file).getroot()
    for line in root.iterfind(".//p:TextLine", ns):
        unicode = line.find("p:TextEquiv/p:Unicode", ns)
        text = unicode.text if unicode is not None and unicode.text else ""
        rows.append({"page": xml_file.stem, "text": text})

endp = pd.DataFrame(rows)
print(len(endp), "lines on", endp["page"].nunique(), "pages")
print(endp.head())

34231 lines on 512 pages
        page                                               text
0  0001_Left                                               260.
1  0001_Left        Lunedi sexta mensis augusti anno nonagesimo
2  0001_Left  secundo que dies lune erat electa ad providend...
3  0001_Left  in facto electionis seu provisionis futuri epi...
4  0001_Left  dominis infrascriptis et convocatis ordinarie ...


In [9]:
def simple(t):
    return " ".join(str(t).lower().split())

catmus_plain = {simple(t) for t in catmus_text if str(t).isascii() and len(str(t)) >= 15}
print(len(catmus_plain), "CATMuS lines without abbreviation marks")

endp["simple"] = endp["text"].map(simple)
matches = endp[endp["simple"].isin(catmus_plain)]
print(len(matches), "matching lines")
print(matches["page"].value_counts())

114 CATMuS lines without abbreviation marks
230 matching lines
page
0009_Left            6
0009_Main_frame      6
0011_Main_frame      6
FRAN_0393_00442_L    6
0008_Main_frame      5
                    ..
FRAN_0393_08201_L    1
FRAN_0393_08202_L    1
FRAN_0393_09893_L    1
FRAN_0393_10227_L    1
FRAN_0393_10228_L    1
Name: count, Length: 114, dtype: int64


In [10]:
pages_per_line = matches.groupby("simple")["page"].nunique().sort_values(ascending=False)
print(pages_per_line.head(15))
print()
print((pages_per_line == 1).sum(), "lines appear on only 1 page")

simple
n. de ordeimonte                                                   67
r. de putheolis                                                    16
gri. cancellario                                                   14
ro. de lorriaco                                                    13
reginaldo de buxia                                                 10
n. chaudart infirmo                                                 9
re. de putheolis                                                    4
gerardo gobaille                                                    2
lapsi anno domini millesimo quadringentesimo nonagesimo secundo     2
ro. de dangello                                                     2
requestam fore racionabilem instanter requirebat illam sibi         2
p. de castro infirmo                                                2
gir. cancellario                                                    2
postulacionem futuri pontificis et diem lune sextam                 2
petro de roch

In [11]:
strong = pages_per_line[pages_per_line <= 2].index
bad_pages = set(matches[matches["simple"].isin(strong)]["page"])
print(len(bad_pages), "pages with strong matches")
print(sorted(bad_pages))

20 pages with strong matches
['0001_Left', '0001_Main_frame', '0002_Main_frame', '0003_Left', '0005_Left', '0007_Left', '0008_Right', '0009_Left', '0010_Right', 'FRAN_0393_00016_L', 'FRAN_0393_00018_L', 'FRAN_0393_00019_L', 'FRAN_0393_00325_L', 'FRAN_0393_00328_L', 'FRAN_0393_00395_L', 'FRAN_0393_00396_L', 'FRAN_0393_00507_L', 'FRAN_0393_08200_L', 'FRAN_0393_08201_L', 'FRAN_0393_08202_L']


In [12]:
pages = pd.Series(sorted(endp["page"].unique()))
fran = pages[pages.str.startswith("FRAN")]
plain = pages[~pages.str.startswith("FRAN")]

print(len(fran), "FRAN pages, from", fran.iloc[0], "to", fran.iloc[-1])
print(len(plain), "plain pages, from", plain.iloc[0], "to", plain.iloc[-1])

458 FRAN pages, from FRAN_0393_00016_L to FRAN_0393_14483_L
54 plain pages, from 0001_Left to LL123_0020_Right


In [13]:
print(plain.tolist())

['0001_Left', '0001_Main_frame', '0002_Main_frame', '0002_Right', '0003_Left', '0003_Main_frame', '0004_Main_frame', '0004_Right', '0005_Left', '0005_Main_frame', '0006_Main_frame', '0006_Right', '0007_Left', '0007_Main_frame', '0008_Main_frame', '0008_Right', '0009_Left', '0009_Main_frame', '0010_Main_frame', '0010_Right', '0011_Left', '0011_Main_frame', '0012_Main_frame', '0012_Right', '0013_Left', '0013_Main_frame', '0014_Main_frame', '0014_Right', '0015_Left', '0016_Right', '0017_Left', '0018_Right', '0019_Left', '0020_Right', 'LL123_0001_Left', 'LL123_0002_Right', 'LL123_0003_Left', 'LL123_0004_Right', 'LL123_0005_Left', 'LL123_0006_Right', 'LL123_0007_Left', 'LL123_0008_Right', 'LL123_0009_Left', 'LL123_0010_Right', 'LL123_0011_Left', 'LL123_0012_Right', 'LL123_0013_Left', 'LL123_0014_Right', 'LL123_0015_Left', 'LL123_0016_Right', 'LL123_0017_Left', 'LL123_0018_Right', 'LL123_0019_Left', 'LL123_0020_Right']


In [14]:
fran_numbers = fran.str.split("_").str[2].astype(int)
bad_numbers = [int(p.split("_")[2]) for p in bad_pages if p.startswith("FRAN")]
print("CATMuS page numbers:", sorted(bad_numbers))

far_enough = [min(abs(n - b) for b in bad_numbers) > 10 for n in fran_numbers]
allowed_fran = fran[far_enough]
print(len(allowed_fran), "of", len(fran), "FRAN pages are allowed")

CATMuS page numbers: [16, 18, 19, 325, 328, 395, 396, 507, 8200, 8201, 8202]
438 of 458 FRAN pages are allowed


In [15]:
rows = []
for page in sorted(allowed_fran):
    root = ET.parse(xml_dir / f"{page}.xml").getroot()
    for line in root.iterfind(".//p:TextLine", ns):
        unicode = line.find("p:TextEquiv/p:Unicode", ns)
        text = unicode.text if unicode is not None and unicode.text else ""
        coords = line.find("p:Coords", ns).get("points")
        if text.strip():
            rows.append({"page": page, "text": text, "coords": coords})

candidates = pd.DataFrame(rows)
sample = candidates.sample(n=200, random_state=42).reset_index(drop=True)

print(len(candidates), "lines on allowed pages")
print(sample["page"].nunique(), "different pages in the sample")
print(sample.head())

AttributeError: 'NoneType' object has no attribute 'get'